In [ ]:
import requests
from lakehouse.daft import bronze, silver
import json
import daft
from deltalake import DeltaTable

In [5]:
CATALOG = "daft_catalog"

# 1. Set Up and Bronze Data

In [6]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [7]:
@daft.udf(return_dtype=daft.DataType.string())
def get_properties(urls: daft.Series) -> list:
    result = []
    for url in urls.to_pylist():
        json_request = requests.get(url).json()
        result.append(json.dumps(json_request["result"]["properties"]))
    return result

In [8]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return daft.from_pylist(results)

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.with_column("properties", get_properties(daft.col("url")))

    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"


bronze_instance = StarWarsBronze(**options)

In [9]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-03-15 22:25:06 | people | execute | Started
2025-03-15 22:25:06 | people | load | Started
2025-03-15 22:25:11 | people | load | Completed in 0.07 min
2025-03-15 22:25:11 | people | transform | Started
2025-03-15 22:25:11 | people | transform | Completed in 0.0 min
2025-03-15 22:25:11 | people | write | Started
c:\Users\nikol\miniconda3\envs\pyspark3-exec\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


                                                           d

2025-03-15 22:25:48 | people | write | Completed in 0.6 min
2025-03-15 22:25:48 | people | execute | Completed in 0.68 min
2025-03-15 22:25:48 | planets | execute | Started
2025-03-15 22:25:48 | planets | load | Started


2025-03-15 22:25:51 | planets | load | Completed in 0.05 min
2025-03-15 22:25:51 | planets | transform | Started
2025-03-15 22:25:51 | planets | transform | Completed in 0.0 min
2025-03-15 22:25:51 | planets | write | Started


                                                           d

2025-03-15 22:26:18 | planets | write | Completed in 0.43 min
2025-03-15 22:26:18 | planets | execute | Completed in 0.5 min


In [10]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8,propertiesUtf8
2025-03-15 22:25:11.691737,Luke Skywalker,1,https://www.swapi.tech/api/people/1,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Luke Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""172"", ""eye_color"": ""blue"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/1""}"
2025-03-15 22:25:11.691737,C-3PO,2,https://www.swapi.tech/api/people/2,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""C-3PO"", ""gender"": ""n/a"", ""skin_color"": ""gold"", ""hair_color"": ""n/a"", ""height"": ""167"", ""eye_color"": ""yellow"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""112BBY"", ""url"": ""https://www.swapi.tech/api/people/2""}"
2025-03-15 22:25:11.691737,R2-D2,3,https://www.swapi.tech/api/people/3,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R2-D2"", ""gender"": ""n/a"", ""skin_color"": ""white, blue"", ""hair_color"": ""n/a"", ""height"": ""96"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/8"", ""birth_year"": ""33BBY"", ""url"": ""https://www.swapi.tech/api/people/3""}"
2025-03-15 22:25:11.691737,Darth Vader,4,https://www.swapi.tech/api/people/4,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Darth Vader"", ""gender"": ""male"", ""skin_color"": ""white"", ""hair_color"": ""none"", ""height"": ""202"", ""eye_color"": ""yellow"", ""mass"": ""136"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""41.9BBY"", ""url"": ""https://www.swapi.tech/api/people/4""}"
2025-03-15 22:25:11.691737,Leia Organa,5,https://www.swapi.tech/api/people/5,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Leia Organa"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""150"", ""eye_color"": ""brown"", ""mass"": ""49"", ""homeworld"": ""https://www.swapi.tech/api/planets/2"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/5""}"
2025-03-15 22:25:11.691737,Owen Lars,6,https://www.swapi.tech/api/people/6,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Owen Lars"", ""gender"": ""male"", ""skin_color"": ""light"", ""hair_color"": ""brown, grey"", ""height"": ""178"", ""eye_color"": ""blue"", ""mass"": ""120"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""52BBY"", ""url"": ""https://www.swapi.tech/api/people/6""}"
2025-03-15 22:25:11.691737,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""Beru Whitesun lars"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""165"", ""eye_color"": ""blue"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""47BBY"", ""url"": ""https://www.swapi.tech/api/people/7""}"
2025-03-15 22:25:11.691737,R5-D4,8,https://www.swapi.tech/api/people/8,"{""created"": ""2025-03-15T00:00:19.864Z"", ""edited"": ""2025-03-15T00:00:19.864Z"", ""name"": ""R5-D4"", ""gender"": ""n/a"", ""skin_color"": ""white, red"", ""hair_color"": ""n/a"", ""height"": ""97"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""unknown"", ""url"": ""https://www.swapi.tech/api/people/8""}"


No. Rows: 82


In [11]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/planets")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8,propertiesUtf8
2025-03-15 22:25:51.851044,Tatooine,1,https://www.swapi.tech/api/planets/1,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""arid"", ""surface_water"": ""1"", ""name"": ""Tatooine"", ""diameter"": ""10465"", ""rotation_period"": ""23"", ""terrain"": ""desert"", ""gravity"": ""1 standard"", ""orbital_period"": ""304"", ""population"": ""200000"", ""url"": ""https://www.swapi.tech/api/planets/1""}"
2025-03-15 22:25:51.851044,Alderaan,2,https://www.swapi.tech/api/planets/2,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""40"", ""name"": ""Alderaan"", ""diameter"": ""12500"", ""rotation_period"": ""24"", ""terrain"": ""grasslands, mountains"", ""gravity"": ""1 standard"", ""orbital_period"": ""364"", ""population"": ""2000000000"", ""url"": ""https://www.swapi.tech/api/planets/2""}"
2025-03-15 22:25:51.851044,Yavin IV,3,https://www.swapi.tech/api/planets/3,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate, tropical"", ""surface_water"": ""8"", ""name"": ""Yavin IV"", ""diameter"": ""10200"", ""rotation_period"": ""24"", ""terrain"": ""jungle, rainforests"", ""gravity"": ""1 standard"", ""orbital_period"": ""4818"", ""population"": ""1000"", ""url"": ""https://www.swapi.tech/api/planets/3""}"
2025-03-15 22:25:51.851044,Hoth,4,https://www.swapi.tech/api/planets/4,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""frozen"", ""surface_water"": ""100"", ""name"": ""Hoth"", ""diameter"": ""7200"", ""rotation_period"": ""23"", ""terrain"": ""tundra, ice caves, mountain ranges"", ""gravity"": ""1.1 standard"", ""orbital_period"": ""549"", ""population"": ""unknown"", ""url"": ""https://www.swapi.tech/api/planets/4""}"
2025-03-15 22:25:51.851044,Dagobah,5,https://www.swapi.tech/api/planets/5,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""murky"", ""surface_water"": ""8"", ""name"": ""Dagobah"", ""diameter"": ""8900"", ""rotation_period"": ""23"", ""terrain"": ""swamp, jungles"", ""gravity"": ""N/A"", ""orbital_period"": ""341"", ""population"": ""unknown"", ""url"": ""https://www.swapi.tech/api/planets/5""}"
2025-03-15 22:25:51.851044,Bespin,6,https://www.swapi.tech/api/planets/6,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""0"", ""name"": ""Bespin"", ""diameter"": ""118000"", ""rotation_period"": ""12"", ""terrain"": ""gas giant"", ""gravity"": ""1.5 (surface), 1 standard (Cloud City)"", ""orbital_period"": ""5110"", ""population"": ""6000000"", ""url"": ""https://www.swapi.tech/api/planets/6""}"
2025-03-15 22:25:51.851044,Endor,7,https://www.swapi.tech/api/planets/7,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""8"", ""name"": ""Endor"", ""diameter"": ""4900"", ""rotation_period"": ""18"", ""terrain"": ""forests, mountains, lakes"", ""gravity"": ""0.85 standard"", ""orbital_period"": ""402"", ""population"": ""30000000"", ""url"": ""https://www.swapi.tech/api/planets/7""}"
2025-03-15 22:25:51.851044,Naboo,8,https://www.swapi.tech/api/planets/8,"{""created"": ""2025-03-15T00:00:19.866Z"", ""edited"": ""2025-03-15T00:00:19.866Z"", ""climate"": ""temperate"", ""surface_water"": ""12"", ""name"": ""Naboo"", ""diameter"": ""12120"", ""rotation_period"": ""26"", ""terrain"": ""grassy hills, swamps, forests, mountains"", ""gravity"": ""1 standard"", ""orbital_period"": ""312"", ""population"": ""4500000000"", ""url"": ""https://www.swapi.tech/api/planets/8""}"


No. Rows: 60


In [12]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Overwrite

In [13]:
config = {
    "load": {
        "mode": "default",
        "filter": "all",
        "date_col": "LH_BronzeTS",
    },
    "transform": {
        "ignore_defaults": False,
        "transformation_order": [
            "add_dummy_col",
            "rename_columns",
            "tbl_transformations",
            "select_columns",
            "cast_column_types",
        ],
        # "tbl_transformations": {"tbl": "custom_transform1"},
        "rename_columns": {
            "planets": {"dummy_col": "dummy"},
            "people": {"dummy_col": "dummy"},
        },
        "select_columns": {
            "planets": ["LH_BronzeTS", "name", "uid", "url", "dummy"],
            "people": ["LH_BronzeTS", "name", "uid", "url", "dummy"],
        },
        "cast_column_types": {
            "planets": {"dummy": "string", "id": "int"},
            "people": {"dummy": "string", "id": "int"},
        },
    },
    "write": {
        "mode": "overwrite",
        "overwrite_schema": True,
    },
}

In [ ]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.where("uid <= '25'")

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        df = df.with_column("uid", daft.col("uid").cast("int"))
        return df

    def source_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.source_schema}/{table}"

    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"

    def add_dummy_col(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.with_column("dummy_col", daft.lit("dummy"))


silver_instance = StarWarsSilver(
    catalog=CATALOG, source_schema="bronze", target_schema="silver", config=config
)

In [15]:
silver_instance.execute("people", "planets")

2025-03-15 22:26:18 | people | execute | Started
2025-03-15 22:26:18 | people | load | Started
2025-03-15 22:26:18 | people | load | Completed in 0.0 min
2025-03-15 22:26:18 | people | transform | Started
2025-03-15 22:26:18 | people | transform | Completed in 0.0 min
2025-03-15 22:26:18 | people | write | Started
2025-03-15 22:26:18 | people | write | Completed in 0.0 min
2025-03-15 22:26:18 | people | execute | Completed in 0.0 min
2025-03-15 22:26:18 | planets | execute | Started
2025-03-15 22:26:18 | planets | load | Started
2025-03-15 22:26:18 | planets | load | Completed in 0.0 min
2025-03-15 22:26:18 | planets | transform | Started
2025-03-15 22:26:18 | planets | transform | Completed in 0.0 min
2025-03-15 22:26:18 | planets | write | Started
2025-03-15 22:26:18 | planets | write | Completed in 0.0 min
2025-03-15 22:26:18 | planets | execute | Completed in 0.0 min


In [16]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/silver/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_SilverTSTimestamp(Microseconds, None)","LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidInt32,urlUtf8,dummyUtf8
2025-03-15 22:26:18.625950,2025-03-15 22:25:11.691737,Luke Skywalker,1,https://www.swapi.tech/api/people/1,dummy
2025-03-15 22:26:18.625950,2025-03-15 22:25:11.691737,C-3PO,2,https://www.swapi.tech/api/people/2,dummy
2025-03-15 22:26:18.625950,2025-03-15 22:25:11.691737,R2-D2,3,https://www.swapi.tech/api/people/3,dummy
2025-03-15 22:26:18.625950,2025-03-15 22:25:11.691737,Darth Vader,4,https://www.swapi.tech/api/people/4,dummy
2025-03-15 22:26:18.625950,2025-03-15 22:25:11.691737,Leia Organa,5,https://www.swapi.tech/api/people/5,dummy
2025-03-15 22:26:18.625950,2025-03-15 22:25:11.691737,Owen Lars,6,https://www.swapi.tech/api/people/6,dummy
2025-03-15 22:26:18.625950,2025-03-15 22:25:11.691737,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7,dummy
2025-03-15 22:26:18.625950,2025-03-15 22:25:11.691737,R5-D4,8,https://www.swapi.tech/api/people/8,dummy


No. Rows: 82


In [17]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/silver/planets")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_SilverTSTimestamp(Microseconds, None)","LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidInt32,urlUtf8,dummyUtf8
2025-03-15 22:26:18.676767,2025-03-15 22:25:51.851044,Tatooine,1,https://www.swapi.tech/api/planets/1,dummy
2025-03-15 22:26:18.676767,2025-03-15 22:25:51.851044,Alderaan,2,https://www.swapi.tech/api/planets/2,dummy
2025-03-15 22:26:18.676767,2025-03-15 22:25:51.851044,Yavin IV,3,https://www.swapi.tech/api/planets/3,dummy
2025-03-15 22:26:18.676767,2025-03-15 22:25:51.851044,Hoth,4,https://www.swapi.tech/api/planets/4,dummy
2025-03-15 22:26:18.676767,2025-03-15 22:25:51.851044,Dagobah,5,https://www.swapi.tech/api/planets/5,dummy
2025-03-15 22:26:18.676767,2025-03-15 22:25:51.851044,Bespin,6,https://www.swapi.tech/api/planets/6,dummy
2025-03-15 22:26:18.676767,2025-03-15 22:25:51.851044,Endor,7,https://www.swapi.tech/api/planets/7,dummy
2025-03-15 22:26:18.676767,2025-03-15 22:25:51.851044,Naboo,8,https://www.swapi.tech/api/planets/8,dummy


No. Rows: 60


In [18]:
dt = DeltaTable(f"D:/Data/{CATALOG}/silver/planets")
daft.from_pylist(dt.history()).show()

clientVersionUtf8,operationUtf8,"operationParametersStruct[location: Utf8, metadata: Utf8, mode: Utf8, protocol: Utf8]",timestampInt64,versionInt64
delta-rs.py-0.25.4,CREATE TABLE,"{location: file:///D:/Data/daft_catalog/silver/planets,metadata: {""configuration"":{},""createdTime"":1742073978698,""description"":null,""format"":{""options"":{},""provider"":""parquet""},""id"":""666c6f6b-0ec3-4fc8-b029-188c555aa45b"",""name"":null,""partitionColumns"":[],""schemaString"":""{\""type\"":\""struct\"",\""fields\"":[{\""name\"":\""LH_SilverTS\"",\""type\"":\""timestamp_ntz\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""LH_BronzeTS\"",\""type\"":\""timestamp_ntz\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""name\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""uid\"",\""type\"":\""integer\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""url\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}},{\""name\"":\""dummy\"",\""type\"":\""string\"",\""nullable\"":true,\""metadata\"":{}}]}""},mode: ErrorIfExists,protocol: {""minReaderVersion"":3,""minWriterVersion"":7,""readerFeatures"":[""timestampNtz""],""writerFeatures"":[""timestampNtz""]},}",1742073978698,0


# 6 Clean Up

In [19]:
import shutil

shutil.rmtree(f"D:/Data/{CATALOG}")